### **1. Data Ingestion**

Loading CSV file using `spark.read` with headers and schema inference.

In [ ]:
# ── CELL 1: INITIALIZE SPARK SESSION ──────────────────────────────────────
import os
import pyspark
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# Build SparkSession locally to avoid Python version mismatch with external cluster
spark = SparkSession.builder \
    .appName("olist-notebook-analysis") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to reduce noise inside the notebook
spark.sparkContext.setLogLevel("WARN")

print("Spark Session created successfully!")
print("Spark Master URI:", spark.sparkContext.master)

🎯 Spark Session created successfully!
Spark Master URI: local[*]


In [2]:
# ==========================================================================
# 1. LOAD CUSTOMERS DATASET FROM MINIO BRONZE LAYER
# ==========================================================================

# Load the customers dataset from the Bronze layer in MinIO
# Delta format automatically handles structure and preserves exact data types
df_customers = (
    spark.read
    .format("delta")
    .load("s3a://bronze/csv/customers/")
)

# 2. Preview the first 10 rows inside the notebook
display(df_customers.limit(10))

DataFrame[customer_id: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, _ingested_at: timestamp, _source_file: string]

### **2. Schema Inspection**
Running `printSchema()` to verify column data types.

In [3]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [4]:
from pyspark.sql.types import StringType, IntegerType
from pyspark.sql import functions as F
# Apply casting to match the silver_customers requirements
df_customers = df_customers \
    .withColumn("customer_id", F.col("customer_id").cast(StringType())) \
    .withColumn("customer_unique_id", F.col("customer_unique_id").cast(StringType())) \
    .withColumn("customer_zip_code_prefix", F.col("customer_zip_code_prefix").cast(IntegerType())) \
    .withColumn("customer_city", F.col("customer_city").cast(StringType())) \
    .withColumn("customer_state", F.col("customer_state").cast(StringType()))

df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



### **3. Integrity Check: Geolocation Mapping**
**Performed** a `left_anti` join between the customers and geolocation tables to calculate the total count and ratio of unmapped customer records.

In [5]:
from pyspark.sql import functions as F
# 1. Read the final processed Geolocation table from the Silver layer in MinIO
df_geo = (
    spark.read
    .format("delta")
    .load("s3a://silver/refined/geolocation/")
)

# 2. Identify customers with missing zip code mapping
# We perform a left_anti join to find customer zip codes that do NOT exist in the geolocation table
unmapped_customers = df_customers.join(
    df_geo, 
    df_customers["customer_zip_code_prefix"] == df_geo["geolocation_zip_code_prefix"], 
    how="left_anti"
)

# 3. Report the Integrity Check
unmapped_count = unmapped_customers.count()
total_customers = df_customers.count()

print("=== CUSTOMER-GEOLOCATION INTEGRITY REPORT ===")
print(f"Total Customers: {total_customers}")
print(f"Customers with Missing Zip Mapping: {unmapped_count}")
if total_customers > 0:
    print(f"Integrity Gap Ratio: {(unmapped_count / total_customers) * 100:.4f}%")
print("="*45)

# 4. Display a sample of orphaned zip codes if any exist
if unmapped_count > 0:
    print("\n=== SAMPLE OF UNMAPPED CUSTOMER ZIP CODES ===")
    unmapped_customers.select("customer_id", "customer_zip_code_prefix").show(15)

=== CUSTOMER-GEOLOCATION INTEGRITY REPORT ===
Total Customers: 99441
Customers with Missing Zip Mapping: 278
Integrity Gap Ratio: 0.2796%

=== SAMPLE OF UNMAPPED CUSTOMER ZIP CODES ===
+--------------------+------------------------+
|         customer_id|customer_zip_code_prefix|
+--------------------+------------------------+
|ecb1725b26e8b8c45...|                   72300|
|bcf86029aeed4ed8b...|                   11547|
|f4302056f0c585705...|                   64605|
|03bbe0ce5c28e05f2...|                   72465|
|ad4950aded55c2ea3...|                    7729|
|6e7dc8f6ec3f0a0d7...|                   72904|
|c55a17a7c31353c35...|                   35408|
|3a9686af66e7ba129...|                   78554|
|baca33004aa726524...|                   73369|
|7557541c9c578082c...|                    8980|
|bbf5e98dabf1bdac7...|                   29949|
|78bebfa74709728a6...|                   65137|
|814dfd64a142fe256...|                   28655|
|5f0099134079adfe1...|                   73255|

### **Integrity Check Results**
* **Observation:** Found 278 unmapped zip codes, creating a 0.2796% data gap.
* **Interpretation:** Minor reference issue with minimal impact on overall data.
* **Action:** Log a copy of unmapped records into the error table and proceed with the main pipeline as normal.

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Customer Data Integrity Audit & Imputation
Identify and audit customer records with missing or unmapped zip codes by validating against the master geolocation dataset. Anomaly reports are generated for quality tracking, and invalid entries are normalized to a '-1' placeholder to maintain relational consistency within the customer data pipeline.

In [6]:
from pyspark.sql import functions as F

# 1. Prepare unique zip codes for lookup
geo_lookup = df_geo.select(F.col("geolocation_zip_code_prefix").alias("geo_zip")).dropDuplicates(["geo_zip"])

# 2. Identify customers with invalid or missing zip codes
# We look for NULLs or ZIPs that don't exist in the geolocation dataset
customers_errors_audit = df_customers.join(
    geo_lookup, 
    df_customers.customer_zip_code_prefix == geo_lookup.geo_zip, 
    how="left"
).filter(
    F.col("customer_zip_code_prefix").isNull() | F.col("geo_zip").isNull()
).withColumn(
    "error_reason", 
    F.when(F.col("customer_zip_code_prefix").isNull(), F.lit("Critical: customer_zip_code_prefix is NULL"))
     .otherwise(F.lit("Orphaned: zip_code not found in geolocation dataset"))
).withColumn(
    "error_detected_at", F.current_timestamp()
).drop("geo_zip") # Removing the lookup join helper

# 3. Clean the main df_customers: Replace invalid ZIPs with -1
# We identify the list of invalid zip codes to update the main dataframe
invalid_zips = [row.customer_zip_code_prefix for row in customers_errors_audit.select("customer_zip_code_prefix").distinct().collect()]

df_customers = df_customers.withColumn(
    "customer_zip_code_prefix",
    F.when(F.col("customer_zip_code_prefix").isNull() | F.col("customer_zip_code_prefix").isin(invalid_zips), F.lit(-1))
     .otherwise(F.col("customer_zip_code_prefix"))
)

# 4. Final Reporting
print(f"Total error records logged in customers_errors_audit: {customers_errors_audit.count()}")
print("df_customers has been updated: NULLs and unmapped ZIPs replaced with -1.")

# Optional: View the structure of the error audit
# customers_errors_audit.show(5)

Total error records logged in customers_errors_audit: 278
df_customers has been updated: NULLs and unmapped ZIPs replaced with -1.


### **6. Data Profiling: Statistical Summary**
Summary statistics are required to audit data distribution and missing values,
So, Execute `describe()` to compute count, mean, stddev, min, and max for customer attributes.

In [7]:
display(df_customers.describe())

DataFrame[summary: string, customer_id: string, customer_unique_id: string, customer_zip_code_prefix: string, customer_city: string, customer_state: string, _source_file: string]

### **7. Duplicate Analysis: Entire Row Verification**
Need to check for absolute duplicate rows across the entire dataset grain.


In [8]:
from pyspark.sql import functions as F

# 1. Calculate duplicates before removing them
total_count = df_customers.count()
distinct_df = df_customers.dropDuplicates()
distinct_count = distinct_df.count()

duplicate_count = total_count - distinct_count

# 2. Print the duplicate report
print(f"--- Data Quality: Duplicates Report ---")
print(f"Total count before removal: {total_count}")
print(f"Total Duplicates removed: {duplicate_count}")
print(f"Duplicates Ratio: {(duplicate_count / total_count) * 100:.2f}%")

# 3. Update the main DataFrame to the cleaned version
df_customers = distinct_df

# 4. Proceed with the geolocation/zip_code validation (as defined in the previous step)


--- Data Quality: Duplicates Report ---
Total count before removal: 99441
Total Duplicates removed: 0
Duplicates Ratio: 0.00%


### **Duplicate Analysis Results**
**Observation:** Found 0 duplicate rows out of 99,441 total records, confirming a 0.00% duplication ratio. The dataset contains no redundant full-row copies, so we can proceed safely with downstream processing since the row-level integrity is completely clean.

### **8. Granularity Analysis: Customer Unique ID Verification**
We compare the total record count against the unique `customer_unique_id` count to check for any duplication. If duplicates exist, the logic groups the data by unique ID to display a sample of these repeated entries for further analysis.

In [9]:
from pyspark.sql import functions as F

# 1. Calculate total rows vs. unique IDs
total_records = df_customers.count()
unique_records = df_customers.select("customer_unique_id").distinct().count()

# 2. Check for duplicates
if total_records == unique_records:
    print("SUCCESS: No duplicates found in 'customer_unique_id'.")
else:
    duplicate_count = total_records - unique_records
    print(f"WARNING: Found {duplicate_count} duplicate entries in 'customer_unique_id'!")
    
    # 3. Optional: Display sample of the duplicates
    df_customers.groupBy("customer_unique_id") \
        .count() \
        .filter("count > 1") \
        .orderBy(F.col("count").desc()) \
        .show(10)

+--------------------+-----+
|  customer_unique_id|count|
+--------------------+-----+
|8d50f5eadf50201cc...|   17|
|3e43e6105506432c9...|    9|
|6469f99c1f9dfae77...|    7|
|ca77025e7201e3b30...|    7|
|1b6c7548a2a1f9037...|    7|
|12f5d6e1cbf93dafd...|    6|
|f0e310a6839dce9de...|    6|
|de34b16117594161a...|    6|
|47c1a3033b8b77b3a...|    6|
|63cfc61cee11cbe30...|    6|
+--------------------+-----+
only showing top 10 rows



### **Granularity Analysis Results**
The check revealed duplicate entries in `customer_unique_id` even though `customer_id` remains unique per row. This is expected because the dataset grain is transactional (per order), meaning a returning customer will naturally have multiple purchase IDs mapped to a single identity. We will maintain this grain as it is for our order analytics.

### **10. Transactional Integrity & Error Isolation**
Cross-references customer logs with `silver_orders` via a `left` join to audit source system discrepancies. 
* Engineers **`has_matching_order`** and **`error_reason`** to tag orphaned records without data loss.
* Automatically isolates all mismatched rows into a dedicated QA table: **`silver_customers_integrity_errors`**.

In [12]:
from pyspark.sql import functions as F

# ==========================================================================
# 1. LOAD ORDERS DATASET FROM MINIO SILVER LAYER
# ==========================================================================

# FIXED: Replaced spark.table with direct MinIO Delta storage path
df_orders = spark.read.format("delta").load("s3a://silver/refined/orders/")

# 1. Get unique customer IDs from the orders table as a verification lookup
orders_lookup = df_orders.select("customer_id").dropDuplicates(["customer_id"])

# PREVENTIVE FIX: Drop metadata from df_customers before the Join to prevent AMBIGUOUS_REFERENCE
df_customers = df_customers.drop("_ingested_at", "_source_file")

# 2. Perform a Left Join (Using F.col for both sides to ensure proper highlighting)
df_customers = df_customers.join(
    orders_lookup.withColumnRenamed("customer_id", "matched_order_id"),
    F.col("customer_id") == F.col("matched_order_id"),
    how="left"
)

# 3. Engineer 'has_matching_order' flag and drop the temporary lookup column
df_customers = df_customers.withColumn(
    "has_matching_order",
    F.when(F.col("matched_order_id").isNotNull(), True).otherwise(False)
).drop("matched_order_id")

# 4. Isolate the unmatched transactions INTO the Integrity Errors DataFrame
df_customers_integrity_errors = df_customers.filter(F.col("has_matching_order") == False)

# 5. Populate 'error_reason' and timestamp ONLY inside the errors dataframe
df_customers_integrity_errors = df_customers_integrity_errors \
    .withColumn("error_reason", F.lit("Orphaned Transaction - Missing Order Record in Source")) \
    .withColumn("error_detected_at", F.current_timestamp())


# ==========================================================================
# 6. PERSIST INTEGRITY ERRORS TO MINIO
# ==========================================================================

# Define the target absolute storage path on MinIO for customers integrity errors
CUSTOMERS_INTEGRITY_ERRORS_PATH = "s3a://silver/qa_issues/silver_customers_integrity_errors/"

# FIXED: Save using direct MinIO S3A paths instead of saveAsTable
df_customers_integrity_errors.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(CUSTOMERS_INTEGRITY_ERRORS_PATH)

# Refresh the Delta cache for this path to ensure instant data governance visibility
spark.catalog.refreshByPath(CUSTOMERS_INTEGRITY_ERRORS_PATH)


# ==========================================================================
# 7. PRODUCTION QUALITY AND PROCESSING REPORT
# ==========================================================================

print("=== GEOLOCATION & INTEGRITY PROCESSING REPORT ===")
# FIXED: Completed the broken line and assigned counts to variables safely
total_count = df_customers.count()
matched_count = df_customers.filter(F.col("has_matching_order") == True).count()
unmatched_count = df_customers_integrity_errors.count()

print(f"Total Transactions Processed (Kept for Geo-Analysis): {total_count}")
print(f"Matched Transactions: {matched_count}")
print(f"Unmatched Transactions (Appended to silver_customers_integrity_errors): {unmatched_count}")
print("="*60)

# 8. Preview the final unified DataFrame (Clean with no error columns)
print("\n=== FINAL UNIFIED CUSTOMERS SAMPLE PREVIEW ===")
df_customers.select("customer_id", "customer_state", "has_matching_order").show(10, truncate=False)

=== GEOLOCATION & INTEGRITY PROCESSING REPORT ===
Total Transactions Processed (Kept for Geo-Analysis): 99441
Matched Transactions: 99441
Unmatched Transactions (Appended to silver_customers_integrity_errors): 0

=== FINAL UNIFIED CUSTOMERS SAMPLE PREVIEW ===
+--------------------------------+--------------+------------------+
|customer_id                     |customer_state|has_matching_order|
+--------------------------------+--------------+------------------+
|4e7b3e00288586ebd08712fdd0374a03|SP            |true              |
|b2d1536598b73a9abd18e0d75d92f0a3|SP            |true              |
|206f3129c0e4d7d0b9550426023f0a08|SP            |true              |
|b6368ca0f56d4632f44d58ca431487b2|SC            |true              |
|f34a6e874087ec1f0e3dab9fdf659c5d|MG            |true              |
|8fcaa9368903f3a9a28aeaff28c14638|MS            |true              |
|cc32707d2e2f7c92ab449f9b28154809|RJ            |true              |
|97e126f19a6f04b3462619f36862bcd2|SP            |t

In [13]:
# Drop the 'has_matching_order' column from df_customers
df_customers = df_customers.drop("has_matching_order")

# Print the schema to confirm the column is gone
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



### **11. Data Completeness: Null Value Analysis**
We loop through all columns in the customers dataset to count the occurrences of missing or null values per attribute. This helps identify any gaps in the dataset that could impact mandatory fields or downstream analysis.

In [14]:
from pyspark.sql import functions as F
null_counts = df_customers.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_customers.columns])

display(null_counts)

DataFrame[customer_id: bigint, customer_unique_id: bigint, customer_zip_code_prefix: bigint, customer_city: bigint, customer_state: bigint]

### **12. Geographical Exploration: Inspecting Cities and States**
We perform a quick exploration of the geographical data fields by counting the total number of unique cities and states present in the dataset. The logic also lists the record counts across all states and displays a sample of unique cities to get a direct look at how the location data is distributed and formatted.

In [15]:
from pyspark.sql import functions as F

# 1. Count of distinct cities and states
unique_cities_count = df_customers.select("customer_city").distinct().count()
unique_states_count = df_customers.select("customer_state").distinct().count()

print("=== GEOLOCATION DOMAIN SUMMARY ===")
print(f"Total Unique Cities: {unique_cities_count}")
print(f"Total Unique States: {unique_states_count}")
print("="*40)

# 2. Display the top 15 states with the most records to see the data distribution
print("\n=== STATES BY RECORD COUNT ===")
df_customers.groupBy("customer_state") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy(F.col("record_count").desc()) \
    .show(27)

# 3. Display a sample of unique cities
print("\n=== SAMPLE OF DISTINCT CITIES ===")
df_customers.select("customer_city").distinct().limit(15).show()

=== GEOLOCATION DOMAIN SUMMARY ===
Total Unique Cities: 4119
Total Unique States: 27

=== STATES BY RECORD COUNT ===
+--------------+------------+
|customer_state|record_count|
+--------------+------------+
|            SP|       41746|
|            RJ|       12852|
|            MG|       11635|
|            RS|        5466|
|            PR|        5045|
|            SC|        3637|
|            BA|        3380|
|            DF|        2140|
|            ES|        2033|
|            GO|        2020|
|            PE|        1652|
|            CE|        1336|
|            PA|         975|
|            MT|         907|
|            MA|         747|
|            MS|         715|
|            PB|         536|
|            PI|         495|
|            RN|         485|
|            AL|         413|
|            SE|         350|
|            TO|         280|
|            RO|         253|
|            AM|         148|
|            AC|          81|
|            AP|          68|
|            

### **13. Structural Integrity: Whitespace and Casing Impact Check**
We evaluate the consistency of the `customer_city` column by comparing the count of current distinct cities against a simulated cleaned version that applies lowercase and trim operations. This comparison helps determine if duplicate categories are being created due to trailing spaces or inconsistent letter casing.

In [16]:
from pyspark.sql import functions as F

# Calculate the count of distinct cities currently vs. after applying standard cleaning (Lower + Trim)
# This reveals data inconsistencies caused by character casing and leading/trailing whitespace
df_customers.select(
    F.countDistinct("customer_city").alias("current_distinct_cities"),
    F.countDistinct(F.lower(F.trim(F.col("customer_city")))).alias("cleaned_distinct_cities")
).show()

+-----------------------+-----------------------+
|current_distinct_cities|cleaned_distinct_cities|
+-----------------------+-----------------------+
|                   4119|                   4119|
+-----------------------+-----------------------+



### **Consistency Check Results**
The check shows that both the current and the cleaned distinct city counts are exactly 4,119. This identical result confirms that the city names are already perfectly standardized and free of any hidden whitespace or casing discrepancies, meaning no further string-cleaning transformations are required for this column.

### **14. Standardization: Mapping State Abbreviations to Full Names**
We create a Spark-compatible map expression using a dictionary of the 27 Brazilian states to replace the two-letter state abbreviations with their full names. The transformation uses a coalesce function to safeguard the data by retaining the original abbreviation if any code fails to match, drops the old column, and renames the updated field back to maintain the schema.

In [17]:
from pyspark.sql import functions as F
from itertools import chain

# 1. Define the mapping dictionary for the 27 Brazilian states
brazil_states_map = {
    'SP': 'São Paulo', 'MG': 'Minas Gerais', 'RJ': 'Rio de Janeiro',
    'RS': 'Rio Grande do Sul', 'PR': 'Paraná', 'SC': 'Santa Catarina',
    'BA': 'Bahia', 'GO': 'Goiás', 'PE': 'Pernambuco', 'ES': 'Espírito Santo',
    'CE': 'Ceará', 'MT': 'Mato Grosso', 'DF': 'Distrito Federal',
    'MS': 'Mato Grosso do Sul', 'PA': 'Pará', 'MA': 'Maranhão',
    'PB': 'Paraíba', 'RN': 'Rio Grande do Norte', 'PI': 'Piauí',
    'AL': 'Alagoas', 'TO': 'Tocantins', 'SE': 'Sergipe',
    'RO': 'Rondônia', 'AM': 'Amazonas', 'AC': 'Acre',
    'AP': 'Amapá', 'RR': 'Roraima'
}

# 2. Convert the dictionary into a Spark-compatible map expression
mapping_expr = F.create_map([F.lit(x) for x in chain(*brazil_states_map.items())])

# 3. Apply the mapping, drop the old column, and rename the new one
# We use coalesce to ensure that if a code is missing, the original value is kept
df_customers_fixed = df_customers \
    .withColumn("customer_state_full", F.coalesce(mapping_expr[F.col("customer_state")], F.col("customer_state"))) \
    .drop("customer_state") \
    .withColumnRenamed("customer_state_full", "customer_state")

# 4. Verify the transformation
print("=== CUSTOMERS BY FULL STATE NAMES ===")
df_customers_fixed.groupBy("customer_state") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy(F.col("record_count").desc()) \
    .show(27, truncate=False)

=== CUSTOMERS BY FULL STATE NAMES ===
+-------------------+------------+
|customer_state     |record_count|
+-------------------+------------+
|São Paulo          |41746       |
|Rio de Janeiro     |12852       |
|Minas Gerais       |11635       |
|Rio Grande do Sul  |5466        |
|Paraná             |5045        |
|Santa Catarina     |3637        |
|Bahia              |3380        |
|Distrito Federal   |2140        |
|Espírito Santo     |2033        |
|Goiás              |2020        |
|Pernambuco         |1652        |
|Ceará              |1336        |
|Pará               |975         |
|Mato Grosso        |907         |
|Maranhão           |747         |
|Mato Grosso do Sul |715         |
|Paraíba            |536         |
|Piauí              |495         |
|Rio Grande do Norte|485         |
|Alagoas            |413         |
|Sergipe            |350         |
|Tocantins          |280         |
|Rondônia           |253         |
|Amazonas           |148         |
|Acre            

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Geographical Regionalization
Enrich the customer dataset by mapping individual state identifiers to their respective Brazilian macro-regions. This categorical grouping facilitates regional trend analysis and improves the granularity of geographic business reporting.

In [18]:
from pyspark.sql import functions as F

# Add the customer_region column using nested CASE WHEN logic
df_customers_fixed = df_customers_fixed.withColumn(
    "customer_region",
    F.when(F.col("customer_state").isin('SP', 'RJ', 'MG', 'ES'), 'Southeast')
     .when(F.col("customer_state").isin('PR', 'RS', 'SC'), 'South')
     .when(F.col("customer_state").isin('BA', 'PE', 'CE', 'RN', 'MA', 'PB', 'AL', 'SE', 'PI'), 'Northeast')
     .when(F.col("customer_state").isin('MT', 'MS', 'GO', 'DF'), 'Central-West')
     .otherwise('North')
)

# Verify the result
print("=== Preview of df_customers with new Region column ===")
df_customers_fixed.select("customer_state", "customer_region").distinct().orderBy("customer_region").show()

=== Preview of df_customers with new Region column ===
+-------------------+---------------+
|     customer_state|customer_region|
+-------------------+---------------+
|     Rio de Janeiro|          North|
|  Rio Grande do Sul|          North|
|   Distrito Federal|          North|
| Mato Grosso do Sul|          North|
|              Amapá|          North|
|              Ceará|          North|
|        Mato Grosso|          North|
|Rio Grande do Norte|          North|
|           Amazonas|          North|
|            Roraima|          North|
|     Santa Catarina|          North|
|             Paraná|          North|
|           Maranhão|          North|
|          Tocantins|          North|
|          São Paulo|          North|
|       Minas Gerais|          North|
|              Goiás|          North|
|     Espírito Santo|          North|
|               Pará|          North|
|         Pernambuco|          North|
+-------------------+---------------+
only showing top 20 rows



### **15. Market Density Analysis: Geographic Percentage Distribution**
We calculate the percentage distribution of records across both states and cities to identify where the customer base is most heavily concentrated. The logic divides the row count of each geographic region by the total transaction count, rounds the result to two decimal places, and orders the output in descending order to highlight the top-performing markets.

In [19]:
from pyspark.sql import functions as F

# 1. Calculate Total Records (Total Transactions)
total_transactions = df_customers_fixed.count()

# 2. Calculate Percentage Distribution for States
state_distribution = df_customers_fixed.groupBy("customer_state") \
    .agg(F.count("*").alias("transaction_count")) \
    .withColumn("percentage", F.round((F.col("transaction_count") / total_transactions) * 100, 2)) \
    .orderBy(F.col("percentage").desc())

# 3. Calculate Percentage Distribution for Cities
city_distribution = df_customers_fixed.groupBy("customer_city") \
    .agg(F.count("*").alias("transaction_count")) \
    .withColumn("percentage", F.round((F.col("transaction_count") / total_transactions) * 100, 2)) \
    .orderBy(F.col("percentage").desc())

# 4. Display Results
print("=== GEOGRAPHIC DISTRIBUTION: STATES ===")
state_distribution.show(27)

print("=== GEOGRAPHIC DISTRIBUTION: CITIES ===")
city_distribution.show(10)

=== GEOGRAPHIC DISTRIBUTION: STATES ===
+-------------------+-----------------+----------+
|     customer_state|transaction_count|percentage|
+-------------------+-----------------+----------+
|          São Paulo|            41746|     41.98|
|     Rio de Janeiro|            12852|     12.92|
|       Minas Gerais|            11635|      11.7|
|  Rio Grande do Sul|             5466|       5.5|
|             Paraná|             5045|      5.07|
|     Santa Catarina|             3637|      3.66|
|              Bahia|             3380|       3.4|
|   Distrito Federal|             2140|      2.15|
|     Espírito Santo|             2033|      2.04|
|              Goiás|             2020|      2.03|
|         Pernambuco|             1652|      1.66|
|              Ceará|             1336|      1.34|
|               Pará|              975|      0.98|
|        Mato Grosso|              907|      0.91|
|           Maranhão|              747|      0.75|
| Mato Grosso do Sul|              715|   

### **Geographic Distribution Results**
The analysis shows a massive concentration of transactions in a few specific regions, with the state of São Paulo alone driving 41.98% of the total volume, followed by Rio de Janeiro at 12.92% and Minas Gerais at 11.7%. On a city level, São Paulo also takes the lead with 15.63% of all records. These results provide a clear look at the regional density of the data, showing that the top three states account for more than 66% of the entire customer footprint.

### **16. Customer Segmentation: Loyalty and Repeat Purchase Analysis**
We create a dedicated customer loyalty summary by grouping the data by `customer_unique_id` and counting the total number of associated orders. The logic then applies conditional segmenting to classify customers based on their engagement history: those with a single purchase are tagged as "New", those with two to four purchases as "Repeat", and any with higher transaction volumes as "Elite".

In [20]:
from pyspark.sql import functions as F

# 1. Create a summary table containing only the customer and their classification
df_customer_loyalty = df_customers_fixed.groupBy("customer_unique_id") \
    .agg(F.count("customer_id").alias("order_count")) \
    .withColumn(
        "loyalty_segment",
        F.when(F.col("order_count") == 1, "New")
         .when((F.col("order_count") >= 2) & (F.col("order_count") <= 4), "Repeat")
         .otherwise("Elite")
    )


In [21]:
display(df_customer_loyalty.limit(10))

DataFrame[customer_unique_id: string, order_count: bigint, loyalty_segment: string]

In [22]:
display(df_customers_fixed.limit(10))

DataFrame[customer_id: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, customer_region: string]

### **17. High-Value Customer Isolation: Elite Segment Analysis**
We filter the segmented customer loyalty dataset to extract only the "Elite" buyers, who have placed 5 or more orders. The logic selects their unique IDs along with their exact purchase counts, ensures uniqueness by applying a distinct operation, and orders them in descending order to identify the most active accounts on the platform.

In [23]:
from pyspark.sql import functions as F

# 1. Filter the dataset to select only the Elite customer segment
top_customers = df_customer_loyalty.filter(F.col("loyalty_segment") == "Elite") \
    .select("customer_unique_id", "order_count") \
    .distinct() \
    .orderBy(F.col("order_count").desc())

# 2. Display the results (Top 20 loyal customers)
print("=== TOP ELITE CUSTOMERS ===")
top_customers.show(20)

# 3. Quick statistics: How many customers fall into the "Elite" category?
elite_count = top_customers.count()
print(f"Total number of Elite customers: {elite_count}")

=== TOP ELITE CUSTOMERS ===
+--------------------+-----------+
|  customer_unique_id|order_count|
+--------------------+-----------+
|8d50f5eadf50201cc...|         17|
|3e43e6105506432c9...|          9|
|6469f99c1f9dfae77...|          7|
|ca77025e7201e3b30...|          7|
|1b6c7548a2a1f9037...|          7|
|12f5d6e1cbf93dafd...|          6|
|f0e310a6839dce9de...|          6|
|de34b16117594161a...|          6|
|47c1a3033b8b77b3a...|          6|
|63cfc61cee11cbe30...|          6|
|dc813062e0fc23409...|          6|
|394ac4de8f3acb142...|          5|
|fe81bb32c243a86b2...|          5|
|56c8638e7c058b98a...|          5|
|b4e4f24de1e8725b7...|          5|
|35ecdf6858edc6427...|          5|
|5e8f38a9a1c023f3d...|          5|
|74cb1ad7e6d567432...|          5|
|4e65032f1f574189f...|          5|
+--------------------+-----------+

Total number of Elite customers: 19


### **Elite Segment Insights**
The analysis reveals that there are exactly 19 customers who qualify for the Elite tier. The top customer leads with 17 orders, followed by individuals with 9 and 7 orders respectively. Isolating this small but highly loyal group allows for targeted behavior tracking and provides clear metrics on the highest end of our repeat-purchase distribution.

### **18. Operational Metrics: Loyalty Segment Share and Percentage Distribution**
We aggregate the segmented dataset by `loyalty_segment` to compute the total number of unique customers within each category. The logic then calculates the overall total of unique customer IDs across the entire platform, divides each segment's count by this total, and rounds the result to two decimal places to provide a precise percentage breakdown of our customer base.

In [24]:
from pyspark.sql import functions as F

# 1. Calculate the count for each loyalty segment
segment_stats = df_customer_loyalty.groupBy("loyalty_segment") \
    .agg(F.countDistinct("customer_unique_id").alias("unique_customer_count"))

# 2. Calculate the total number of unique customers to estimate the percentage
total_unique_customers = df_customer_loyalty.select("customer_unique_id").distinct().count()

# 3. Add the percentage column
segment_stats = segment_stats.withColumn(
    "percentage", 
    F.round((F.col("unique_customer_count") / total_unique_customers) * 100, 2)
)

# 4. Display the results
print("=== CUSTOMER LOYALTY SEGMENTATION STATS ===")
segment_stats.orderBy(F.col("percentage").desc()).show()

=== CUSTOMER LOYALTY SEGMENTATION STATS ===
+---------------+---------------------+----------+
|loyalty_segment|unique_customer_count|percentage|
+---------------+---------------------+----------+
|            New|                93099|     96.88|
|         Repeat|                 2978|       3.1|
|          Elite|                   19|      0.02|
+---------------+---------------------+----------+



### **Loyalty Distribution Results**
The metrics show an overwhelming concentration in the "New" tier, which accounts for 96.88% of the user base with 93,099 unique customers. The "Repeat" segment makes up 3.1% (2,978 customers), while the "Elite" tier consists of just 19 customers, representing a minor 0.02% share. This distribution highlights a business model heavily driven by one-time acquisition, indicating a clear opportunity to focus on retention strategies to convert more single-buyer accounts into returning users.

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Data Quality Assurance: Placeholder Integration
Append a synthetic "Unknown" record to the customer dataset to serve as a designated fallback for missing or unmapped values. This step ensures referential integrity during relational joins and prevents runtime null-handling errors in downstream analytical models.

In [25]:
from pyspark.sql import Row

# 1. Define the dummy record structure
# We ensure the data types match the final schema
dummy_record = Row(
    customer_id="-1",
    customer_unique_id="-1",
    customer_zip_code_prefix=-1,
    customer_city="Unknown",
    customer_state="Unknown",
    customer_region="Unknown"
)

# 2. Create a DataFrame from the dummy record
dummy_df = spark.createDataFrame([dummy_record])

# 3. Append the dummy record to the final customers DataFrame
df_customers_fixed = df_customers_fixed.unionByName(dummy_df)

print("✅ Dummy record (-1) added successfully to df_customers_final.")

✅ Dummy record (-1) added successfully to df_customers_final.


@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

### Schema Finalization: Customers Table
Enforce a production-ready schema for the `silver_customers` table by explicitly casting identifiers, geographic codes, and descriptive attributes to their required data types. This ensures consistent data structures for downstream modeling and relational integrity.

In [26]:
from pyspark.sql.types import StringType, IntegerType

# Perform the final casting for the silver_customers table
df_customers_fixed = df_customers_fixed \
    .withColumn("customer_id", F.col("customer_id").cast(StringType())) \
    .withColumn("customer_unique_id", F.col("customer_unique_id").cast(StringType())) \
    .withColumn("customer_zip_code_prefix", F.col("customer_zip_code_prefix").cast(IntegerType())) \
    .withColumn("customer_city", F.col("customer_city").cast(StringType())) \
    .withColumn("customer_state", F.col("customer_state").cast(StringType())) \
    .withColumn("customer_region", F.col("customer_region").cast(StringType()))

# Verification of the final schema
print("=== Final Schema for silver_customers ===")
df_customers_fixed.printSchema()

=== Final Schema for silver_customers ===
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_region: string (nullable = true)



### **19. Persistence and Lakehouse Integration (Silver Layer)**


In [27]:
# ==========================================================================
# FINAL PERSISTENCE: SAVING REFINED SILVER CUSTOMERS
# ==========================================================================

# Define the target absolute storage paths on MinIO
SILVER_CUSTOMERS_DELTA_PATH   = "s3a://silver/refined/customers/"
SILVER_CUSTOMERS_PARQUET_PATH = "s3a://silver/refined/customers_parquet/"

# FIX FOR MINIO/DELTA: Drop ambiguous metadata columns to prevent duplicate columns error during save
df_customers_fixed_cleaned = df_customers_fixed.drop("_ingested_at", "_source_file")

# 1. Save as a Delta Table using direct MinIO S3A paths (FIXED: replaced saveAsTable)
df_customers_fixed_cleaned.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(SILVER_CUSTOMERS_DELTA_PATH)

# Refresh the Delta cache for immediate query capability inside the cluster
spark.catalog.refreshByPath(SILVER_CUSTOMERS_DELTA_PATH)


# 2. Export as Parquet files to the Silver directory on MinIO (FIXED: updated local path to S3A)
df_customers_fixed_cleaned.write \
    .mode("overwrite") \
    .parquet(SILVER_CUSTOMERS_PARQUET_PATH)

# Final confirmation log
print("Done! Silver Customers table is saved as Delta and Parquet on MinIO.")

Done! Silver Customers table is saved as Delta and Parquet on MinIO.


In [28]:
# ==========================================================================
# AUDIT PERSISTENCE: SAVING CUSTOMERS QUALITY AUDIT LOGS
# ==========================================================================

# Define the absolute MinIO S3A storage path for the customers audit logs
CUSTOMERS_AUDIT_LOG_PATH = "s3a://silver/qa_issues/silver_customers_errors/"

# FIX FOR MINIO/DELTA: Drop ambiguous metadata columns from audit dataframe before persistence
customers_errors_audit_cleaned = customers_errors_audit.drop("_ingested_at", "_source_file")

print(f"Saving data quality audit logs to MinIO path: {CUSTOMERS_AUDIT_LOG_PATH}...")

# Save the customers_errors_audit DataFrame as a Delta table (FIXED: replaced saveAsTable)
# We use 'append' mode to keep a historical log of all errors found
customers_errors_audit_cleaned.write \
    .mode("append") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .save(CUSTOMERS_AUDIT_LOG_PATH)

# Explicitly refresh the Delta cache for this path to ensure instant data governance visibility
spark.catalog.refreshByPath(CUSTOMERS_AUDIT_LOG_PATH)

print(f"Success! silver_customers_errors_audit table saved successfully to MinIO path: {CUSTOMERS_AUDIT_LOG_PATH}")

Saving data quality audit logs to MinIO path: s3a://silver/qa_issues/silver_customers_errors/...
Success! silver_customers_errors_audit table saved successfully to MinIO path: s3a://silver/qa_issues/silver_customers_errors/
